## 0. Setup (run once)

In [ ]:
!pip install -q sentence-transformers faiss-cpu chromadb pypdf langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
import os, glob, uuid, shutil, warnings
import numpy as np
warnings.filterwarnings("ignore")

import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

# ---- Models (change LLM_NAME to "google/flan-t5-large" for better answers) ----
EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_NAME = "google/flan-t5-base"

device = 0 if torch.cuda.is_available() else -1
print("Using", "GPU" if device == 0 else "CPU")

embedder = SentenceTransformer(EMBED_NAME)

# For text-generation pipelines with Seq2Seq models in newer transformers, we explicitly load the model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME)
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
qa_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

def llm_generate(prompt, max_new_tokens=200):
    out = qa_pipe(prompt, max_new_tokens=max_new_tokens, truncation=True, do_sample=False)
    generated = out[0]["generated_text"]
    if generated.startswith(prompt):
        generated = generated[len(prompt):]
    return generated.strip()

# ---- Sample documents used by experiments 6-10 ----
SAMPLE_DIR = "/content/sample_docs"
os.makedirs(SAMPLE_DIR, exist_ok=True)

SAMPLES = {
"machine_learning.txt": """Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning and reinforcement learning.

In supervised learning, a model is trained on labeled examples, such as emails marked as spam or not spam. Common supervised algorithms include linear regression, decision trees and support vector machines. In unsupervised learning, the model finds structure in unlabeled data, for example by clustering customers into groups using the k-means algorithm.

Reinforcement learning trains an agent to make decisions by rewarding good actions and penalising bad ones. It is used in robotics and game playing. Deep learning is a subfield of machine learning that uses neural networks with many layers. Deep learning powers image recognition, speech recognition and modern language models.

Overfitting happens when a model memorises the training data and performs poorly on new data. Techniques such as cross-validation, regularisation and dropout help reduce overfitting. The transformer architecture, introduced in 2017 in the paper Attention Is All You Need, is the foundation of most large language models today.""",

"rag_and_vector_databases.txt": """Retrieval-Augmented Generation, or RAG, is a technique that improves the answers of a large language model by retrieving relevant documents and adding them to the prompt. RAG was introduced by Patrick Lewis and colleagues at Facebook AI Research in 2020. It reduces hallucination because the model answers from retrieved evidence instead of relying only on its memory.

A typical RAG pipeline has these steps: load documents, split them into chunks, convert each chunk into an embedding, store the embeddings in a vector database, retrieve the top-k most similar chunks for a user question, and pass them to the language model to generate the final answer.

An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meaning have embeddings that are close together. Cosine similarity measures the angle between two embedding vectors and is the most common way to compare them.

FAISS stands for Facebook AI Similarity Search. It is an open-source library developed by Meta AI for fast similarity search over dense vectors, and it supports GPU acceleration. ChromaDB is an open-source embedding database that stores documents, embeddings and metadata, and it is popular for building LLM applications. Other vector databases include Pinecone, Weaviate and Milvus.

Chunk size matters in RAG. Chunks that are too large add noise, while chunks that are too small lose context. A chunk size of 300 to 1000 characters with a small overlap is a common starting point.""",

"climate_change.txt": """Climate change refers to long-term shifts in global temperatures and weather patterns. Since the 1800s, human activities have been the main driver of climate change, primarily because of the burning of fossil fuels such as coal, oil and gas. Burning fossil fuels releases greenhouse gases such as carbon dioxide and methane, which trap heat in the atmosphere.

The Paris Agreement, adopted in 2015, is an international treaty on climate change. Its goal is to limit global warming to well below 2 degrees Celsius above pre-industrial levels, and to pursue efforts to limit the increase to 1.5 degrees Celsius.

Renewable energy sources such as solar, wind and hydropower produce electricity without releasing carbon dioxide. Many countries are investing in renewable energy to reduce emissions. Other solutions include improving energy efficiency, protecting forests and adopting electric vehicles.

The effects of climate change include rising sea levels, more frequent heatwaves, stronger storms and changes in rainfall patterns. Coastal cities and small island nations are especially vulnerable to rising sea levels.""",
}
for name, text in SAMPLES.items():
    with open(os.path.join(SAMPLE_DIR, name), "w", encoding="utf-8") as f:
        f.write(text)
print("Sample documents written to", SAMPLE_DIR, "->", os.listdir(SAMPLE_DIR))

Using CPU


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCa

Sample documents written to /content/sample_docs -> ['machine_learning.txt', 'rag_and_vector_databases.txt', 'climate_change.txt']


---
## Experiment 1: Generate text embeddings and perform semantic similarity search
**Idea:** convert sentences into dense vectors, then find the sentences closest in meaning to a query.

In [ ]:
from sentence_transformers import util

sentences = [
    "Machine learning lets computers learn patterns from data.",
    "Deep learning uses neural networks with many layers.",
    "The cat sat quietly on the warm windowsill.",
    "Dogs are loyal pets and love to play fetch.",
    "Python is a popular programming language for data science.",
    "The stock market fell sharply after the announcement.",
    "Investors are worried about rising interest rates.",
    "Pasta carbonara is made with eggs, cheese and pepper.",
]

# 1. Generate embeddings
embeddings = embedder.encode(sentences, convert_to_numpy=True, show_progress_bar=False)
print("Embedding matrix shape:", embeddings.shape, "(sentences x dimensions)")
print("First 8 values of sentence 0:", np.round(embeddings[0][:8], 4))

# 2. Semantic similarity search
def semantic_search(query, top_k=3):
    q_emb = embedder.encode(query, convert_to_numpy=True)
    hits = util.semantic_search(q_emb, embeddings, top_k=top_k)[0]
    return [(sentences[h["corpus_id"]], h["score"]) for h in hits]

for q in ["How do neural networks learn?", "My pet is playing outside", "Financial markets are dropping", "What should I cook for dinner?"]:
    print(f"\nQuery: {q}")
    for text, score in semantic_search(q):
        print(f"  {score:.3f}  {text}")

Embedding matrix shape: (8, 384) (sentences x dimensions)
First 8 values of sentence 0: [-0.0522 -0.0087  0.0631 -0.0056  0.0062 -0.0182 -0.009  -0.0914]

Query: How do neural networks learn?
  0.572  Deep learning uses neural networks with many layers.
  0.522  Machine learning lets computers learn patterns from data.
  0.124  Dogs are loyal pets and love to play fetch.

Query: My pet is playing outside
  0.449  Dogs are loyal pets and love to play fetch.
  0.251  The cat sat quietly on the warm windowsill.
  0.053  The stock market fell sharply after the announcement.

Query: Financial markets are dropping
  0.565  The stock market fell sharply after the announcement.
  0.416  Investors are worried about rising interest rates.
  0.066  Python is a popular programming language for data science.

Query: What should I cook for dinner?
  0.296  Pasta carbonara is made with eggs, cheese and pepper.
  0.088  Dogs are loyal pets and love to play fetch.
  0.065  The cat sat quietly on the wa

---
## Experiment 2: Semantic search system using cosine similarity
**Idea:** implement cosine similarity from scratch (`cos(a,b) = a.b / (|a||b|)`) and wrap it in a small search engine class.

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def cosine_similarity_matrix(A, B):
    A_n = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_n = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_n @ B_n.T          # shape: (len(A), len(B))

class SemanticSearchEngine:
    def __init__(self, model):
        self.model = model
        self.documents = []
        self.doc_embeddings = None

    def index(self, documents):
        self.documents = list(documents)
        self.doc_embeddings = self.model.encode(self.documents, convert_to_numpy=True, show_progress_bar=False)
        print(f"Indexed {len(self.documents)} documents.")

    def search(self, query, top_k=3):
        q_emb = self.model.encode([query], convert_to_numpy=True)
        sims = cosine_similarity_matrix(q_emb, self.doc_embeddings)[0]
        top_idx = np.argsort(-sims)[:top_k]
        return [(int(i), self.documents[i], float(sims[i])) for i in top_idx]

docs = [
    "The Eiffel Tower is a famous landmark in Paris, France.",
    "Photosynthesis is how plants convert sunlight into chemical energy.",
    "The Python language was created by Guido van Rossum.",
    "Mount Everest is the tallest mountain above sea level.",
    "Neural networks are inspired by the structure of the human brain.",
    "The Great Wall of China is thousands of kilometres long.",
    "Plants use chlorophyll to absorb light for growth.",
    "Java and C++ are widely used programming languages.",
]

engine = SemanticSearchEngine(embedder)
engine.index(docs)

for q in ["Who invented Python?", "How do plants make food?", "Tall mountains of the world"]:
    print(f"\nQuery: {q}")
    for rank, (i, text, score) in enumerate(engine.search(q, top_k=3), 1):
        print(f"  #{rank}  score={score:.3f}  [doc {i}] {text}")

# Sanity check: our manual cosine matches the library's value
a, b = engine.doc_embeddings[1], engine.doc_embeddings[6]
print("\nManual cosine :", round(cosine_similarity(a, b), 4))
print("Library cosine:", round(float(util.cos_sim(a, b)), 4))

Indexed 8 documents.

Query: Who invented Python?
  #1  score=0.712  [doc 2] The Python language was created by Guido van Rossum.
  #2  score=0.309  [doc 7] Java and C++ are widely used programming languages.
  #3  score=0.151  [doc 4] Neural networks are inspired by the structure of the human brain.

Query: How do plants make food?
  #1  score=0.510  [doc 6] Plants use chlorophyll to absorb light for growth.
  #2  score=0.492  [doc 1] Photosynthesis is how plants convert sunlight into chemical energy.
  #3  score=0.083  [doc 7] Java and C++ are widely used programming languages.

Query: Tall mountains of the world
  #1  score=0.588  [doc 3] Mount Everest is the tallest mountain above sea level.
  #2  score=0.346  [doc 5] The Great Wall of China is thousands of kilometres long.
  #3  score=0.159  [doc 0] The Eiffel Tower is a famous landmark in Paris, France.

Manual cosine : 0.6246
Library cosine: 0.6246


---
## Experiment 3: Vector database with FAISS and ChromaDB
**Idea:** store embeddings in a vector index/database and retrieve similar documents quickly.

In [ ]:
import faiss, chromadb

corpus = [
    "Python is widely used for machine learning and data analysis.",
    "TensorFlow and PyTorch are popular deep learning frameworks.",
    "The Amazon rainforest is the largest tropical rainforest on Earth.",
    "Coral reefs are threatened by rising ocean temperatures.",
    "Cricket is a bat-and-ball game played between two teams of eleven players.",
    "The FIFA World Cup is the biggest football tournament in the world.",
    "Vitamin C is found in citrus fruits such as oranges and lemons.",
    "Regular exercise improves heart health and mood.",
    "Quantum computers use qubits instead of classical bits.",
    "Electric vehicles reduce tailpipe emissions compared to petrol cars.",
]
doc_emb = embedder.encode(corpus, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
dim = doc_emb.shape[1]
print("Vectors:", doc_emb.shape)

# ---------------- FAISS ----------------
# With normalised vectors, inner product == cosine similarity
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(doc_emb)
print("FAISS index size:", faiss_index.ntotal)

def faiss_search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = faiss_index.search(q, k)
    return [(corpus[i], float(s)) for i, s in zip(ids[0], scores[0])]

print("\n=== FAISS ===")
for q in ["deep learning libraries", "football competition", "healthy lifestyle"]:
    print(f"Query: {q}")
    for text, s in faiss_search(q):
        print(f"  {s:.3f}  {text}")

faiss.write_index(faiss_index, "faiss_index.bin")        # save to disk
faiss_index = faiss.read_index("faiss_index.bin")        # load back
print("Reloaded FAISS index with", faiss_index.ntotal, "vectors")

# ---------------- ChromaDB ----------------
chroma_client = chromadb.Client()                         # in-memory
try:
    chroma_client.delete_collection("exp3_docs")
except Exception:
    pass
collection = chroma_client.create_collection("exp3_docs", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[f"doc{i}" for i in range(len(corpus))],
    documents=corpus,
    embeddings=doc_emb.tolist(),
)
print("\n=== ChromaDB === documents stored:", collection.count())

def chroma_search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q, n_results=k)
    # cosine distance = 1 - cosine similarity
    return [(d, 1 - dist) for d, dist in zip(res["documents"][0], res["distances"][0])]

for q in ["deep learning libraries", "football competition", "healthy lifestyle"]:
    print(f"Query: {q}")
    for text, s in chroma_search(q):
        print(f"  {s:.3f}  {text}")

Vectors: (10, 384)
FAISS index size: 10

=== FAISS ===
Query: deep learning libraries
  0.710  TensorFlow and PyTorch are popular deep learning frameworks.
  0.355  Python is widely used for machine learning and data analysis.
  0.089  Quantum computers use qubits instead of classical bits.
Query: football competition
  0.459  The FIFA World Cup is the biggest football tournament in the world.
  0.290  Cricket is a bat-and-ball game played between two teams of eleven players.
  0.061  The Amazon rainforest is the largest tropical rainforest on Earth.
Query: healthy lifestyle
  0.407  Regular exercise improves heart health and mood.
  0.118  Vitamin C is found in citrus fruits such as oranges and lemons.
  0.038  The FIFA World Cup is the biggest football tournament in the world.
Reloaded FAISS index with 10 vectors

=== ChromaDB === documents stored: 10
Query: deep learning libraries
  0.710  TensorFlow and PyTorch are popular deep learning frameworks.
  0.355  Python is widely used fo

---
## Experiment 4: Document storage and top-k retrieval using a vector database
**Idea:** a persistent store with metadata, top-k retrieval, metadata filtering, update and delete.

In [ ]:
import chromadb

class DocumentStore:
    """Persistent document store backed by ChromaDB."""
    def __init__(self, path="/content/chroma_exp4", name="documents"):
        self.client = chromadb.PersistentClient(path=path)
        self.col = self.client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})

    def add_documents(self, ids, texts, metadatas=None):
        emb = embedder.encode(texts, normalize_embeddings=True).tolist()
        self.col.upsert(ids=ids, documents=texts, embeddings=emb, metadatas=metadatas)   # upsert = safe to re-run

    def top_k(self, query, k=3, where=None):
        q = embedder.encode([query], normalize_embeddings=True).tolist()
        res = self.col.query(query_embeddings=q, n_results=min(k, self.col.count()), where=where)
        return [
            {"id": i, "text": d, "meta": m, "similarity": round(1 - dist, 3)}
            for i, d, m, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])
        ]

    def delete(self, ids):
        self.col.delete(ids=ids)

    def count(self):
        return self.col.count()

store = DocumentStore()

texts = [
    "Lionel Messi won the 2022 FIFA World Cup with Argentina.",
    "The Indian Premier League is a popular T20 cricket league.",
    "Wimbledon is the oldest tennis tournament in the world.",
    "GPUs accelerate the training of deep neural networks.",
    "Large language models are trained on huge text datasets.",
    "Transformers use self-attention to process sequences.",
    "The Reserve Bank controls monetary policy and interest rates.",
    "Inflation measures how fast prices rise over time.",
]
metas = [{"category": "sports"}] * 3 + [{"category": "ai"}] * 3 + [{"category": "economics"}] * 2
store.add_documents([f"d{i}" for i in range(len(texts))], texts, metas)
print("Documents in store:", store.count())

def show(results):
    for r in results:
        print(f"  sim={r['similarity']:.3f} [{r['meta']['category']}] {r['text']}")

print("\nTop-3 for 'how are language models trained?'")
show(store.top_k("how are language models trained?", k=3))

print("\nTop-1 vs Top-5 for 'tennis':")
show(store.top_k("tennis", k=1)); print("  ---"); show(store.top_k("tennis", k=5))

print("\nTop-2 restricted to category='economics' for 'rising prices':")
show(store.top_k("rising prices", k=2, where={"category": "economics"}))

store.delete(["d7"])
print("\nAfter deleting d7 ->", store.count(), "documents")

# Persistence check: open the same folder with a brand-new client
store2 = DocumentStore()
print("Reopened persistent store, documents found:", store2.count())

Documents in store: 8

Top-3 for 'how are language models trained?'
  sim=0.688 [ai] Large language models are trained on huge text datasets.
  sim=0.274 [ai] GPUs accelerate the training of deep neural networks.
  sim=0.248 [ai] Transformers use self-attention to process sequences.

Top-1 vs Top-5 for 'tennis':
  sim=0.601 [sports] Wimbledon is the oldest tennis tournament in the world.
  ---
  sim=0.601 [sports] Wimbledon is the oldest tennis tournament in the world.
  sim=0.198 [sports] The Indian Premier League is a popular T20 cricket league.
  sim=0.109 [sports] Lionel Messi won the 2022 FIFA World Cup with Argentina.
  sim=0.080 [ai] GPUs accelerate the training of deep neural networks.
  sim=0.071 [ai] Large language models are trained on huge text datasets.

Top-2 restricted to category='economics' for 'rising prices':
  sim=0.579 [economics] Inflation measures how fast prices rise over time.
  sim=0.222 [economics] The Reserve Bank controls monetary policy and interest rates.

---
## Experiment 5: Document Question Answering system using RAG
**Idea:** retrieve the most relevant passages, put them in a prompt, and let the LLM answer from that context only.

In [ ]:
import chromadb

knowledge_base = [
    "The Taj Mahal is a white marble mausoleum in Agra, India. It was commissioned by Emperor Shah Jahan in 1632 in memory of his wife Mumtaz Mahal.",
    "The Great Barrier Reef is the world's largest coral reef system, located off the coast of Queensland, Australia.",
    "Python was created by Guido van Rossum and first released in 1991. It emphasises code readability.",
    "The human heart has four chambers: two atria and two ventricles. It pumps blood throughout the body.",
    "The Moon orbits the Earth roughly every 27 days. Its gravity causes the ocean tides.",
    "The Indian Space Research Organisation (ISRO) launched the Chandrayaan-3 mission, which landed near the lunar south pole in August 2023.",
    "Water boils at 100 degrees Celsius at sea level atmospheric pressure.",
    "Mahatma Gandhi led the Indian independence movement using nonviolent civil disobedience.",
]

client5 = chromadb.Client()
try:
    client5.delete_collection("exp5_kb")
except Exception:
    pass
kb_col = client5.create_collection("exp5_kb", metadata={"hnsw:space": "cosine"})
kb_col.add(
    ids=[f"kb{i}" for i in range(len(knowledge_base))],
    documents=knowledge_base,
    embeddings=embedder.encode(knowledge_base, normalize_embeddings=True).tolist(),
)

def retrieve(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True).tolist()
    res = kb_col.query(query_embeddings=q, n_results=k)
    return res["documents"][0]

PROMPT_5 = """Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
Answer:"""

def rag_answer(question, k=3, show_context=True):
    passages = retrieve(question, k)
    context = "\n".join(f"- {p}" for p in passages)
    answer = llm_generate(PROMPT_5.format(context=context, question=question))
    print("Q:", question)
    if show_context:
        print("Retrieved context:")
        for p in passages:
            print("   *", p[:110] + ("..." if len(p) > 110 else ""))
    print("A:", answer, "\n")
    return answer

for q in [
    "Who built the Taj Mahal and why?",
    "Who created Python?",
    "How many chambers does the human heart have?",
    "Which mission landed near the lunar south pole?",
    "Who won the 2010 football world cup?",      # not in the documents
]:
    rag_answer(q)

Q: Who built the Taj Mahal and why?
Retrieved context:
   * The Taj Mahal is a white marble mausoleum in Agra, India. It was commissioned by Emperor Shah Jahan in 1632 in...
   * Mahatma Gandhi led the Indian independence movement using nonviolent civil disobedience.
   * The Indian Space Research Organisation (ISRO) launched the Chandrayaan-3 mission, which landed near the lunar ...
A:  

Q: Who created Python?
Retrieved context:
   * Python was created by Guido van Rossum and first released in 1991. It emphasises code readability.
   * Water boils at 100 degrees Celsius at sea level atmospheric pressure.
   * Mahatma Gandhi led the Indian independence movement using nonviolent civil disobedience.
A:  

Q: How many chambers does the human heart have?
Retrieved context:
   * The human heart has four chambers: two atria and two ventricles. It pumps blood throughout the body.
   * Water boils at 100 degrees Celsius at sea level atmospheric pressure.
   * The Great Barrier Reef is the wor

---
### LangChain setup (shared by Experiments 6 to 10)
Defines document loaders, a text splitter, embeddings, a Chroma vector store builder and the LLM wrapper.

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

lc_embeddings = HuggingFaceEmbeddings(model_name=EMBED_NAME, encode_kwargs={"normalize_embeddings": True})
llm = HuggingFacePipeline(pipeline=qa_pipe, pipeline_kwargs={"max_new_tokens": 200})

def load_file(path):
    """Load a single .txt or .pdf file into LangChain Documents."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        docs = TextLoader(path, encoding="utf-8").load()
    elif ext == ".pdf":
        docs = PyPDFLoader(path).load()
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    for d in docs:
        d.metadata["doc_name"] = os.path.basename(path)
    return docs

def load_folder(folder):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*"))):
        if path.lower().endswith((".txt", ".pdf")):
            docs.extend(load_file(path))
    return docs

def split_documents(docs, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

def new_vectorstore(name="kb"):
    return Chroma(
        collection_name=f"{name}_{uuid.uuid4().hex[:8]}",       # unique name so re-running never collides
        embedding_function=lc_embeddings,
        collection_metadata={"hnsw:space": "cosine"},
    )

def build_vectorstore(chunks, name="kb"):
    vs = new_vectorstore(name)
    vs.add_documents(chunks)
    return vs

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

RAG_PROMPT = PromptTemplate.from_template(
"""Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
Answer:""")
print("LangChain components ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LangChain components ready.


---
## Experiment 6: End-to-end RAG pipeline
**Pipeline:** document loading, text chunking, embeddings, vector store, retrieval, answer generation.

In [ ]:
class RAGPipeline:
    def __init__(self, folder, chunk_size=500, chunk_overlap=50, k=3):
        self.k = k
        # 1. Load
        self.docs = load_folder(folder)
        print(f"[1] Loaded {len(self.docs)} document(s) from {folder}")
        # 2. Chunk
        self.chunks = split_documents(self.docs, chunk_size, chunk_overlap)
        print(f"[2] Split into {len(self.chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")
        # 3+4. Embed and store
        self.vs = build_vectorstore(self.chunks, "exp6")
        print(f"[3] Embedded chunks with {EMBED_NAME}")
        print(f"[4] Stored {len(self.chunks)} vectors in Chroma")
        self.retriever = self.vs.as_retriever(search_kwargs={"k": k})
        # 5+6. Retrieval + generation chain
        self.chain = (
            {"context": self.retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
            | RAG_PROMPT | llm | StrOutputParser()
        )

    def ask(self, question):
        sources = self.retriever.invoke(question)
        answer = self.chain.invoke(question)
        return answer.strip(), sources

pipeline_6 = RAGPipeline(SAMPLE_DIR)

print("\nSample chunk:\n", pipeline_6.chunks[0].page_content[:300], "...\n")

for q in [
    "Who introduced RAG and when?",
    "What does FAISS stand for?",
    "What is the goal of the Paris Agreement?",
    "What is overfitting?",
]:
    answer, sources = pipeline_6.ask(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", sorted({s.metadata["doc_name"] for s in sources}), "\n")

[1] Loaded 3 document(s) from /content/sample_docs
[2] Split into 13 chunks (size=500, overlap=50)
[3] Embedded chunks with sentence-transformers/all-MiniLM-L6-v2
[4] Stored 13 vectors in Chroma

Sample chunk:
 Climate change refers to long-term shifts in global temperatures and weather patterns. Since the 1800s, human activities have been the main driver of climate change, primarily because of the burning of fossil fuels such as coal, oil and gas. Burning fossil fuels releases greenhouse gases such as car ...

Q: Who introduced RAG and when?
A: Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
Retrieval-Augmented Generation, or RAG, is a technique that improves the answers of a large language model by retrieving relevant documents and adding them to the prompt. RAG was introduced by Patrick Lewis and colleagues at Facebook AI Research in 2020. It reduces hallucination because the model answers from retrieved evidence i

---
## Experiment 7: Domain-specific chatbot using LangChain and a vector database
**Domain:** a university admissions helpdesk. The bot answers only from its domain knowledge base and politely refuses off-topic questions (using a similarity threshold).

*Edit the `faq_data` list to switch to any other domain (hospital, bank, e-commerce, ...).*

In [ ]:
faq_data = [
    ("Admission eligibility", "To apply for the B.Tech programme, a candidate must have completed 12th grade with Physics, Chemistry and Mathematics and scored at least 60 percent."),
    ("Application deadline", "The last date to submit the online application for the 2026 academic year is 30 June 2026. Late applications are accepted until 10 July with a penalty fee."),
    ("Application fee", "The application fee is 1000 rupees for general category candidates and 500 rupees for reserved category candidates. It can be paid online by card, UPI or net banking."),
    ("Documents required", "Applicants must upload their 10th and 12th mark sheets, a transfer certificate, a passport-size photograph, an ID proof and a category certificate if applicable."),
    ("Tuition fees", "The annual tuition fee for B.Tech is 1.2 lakh rupees. Hostel and mess charges are 80000 rupees per year, billed separately."),
    ("Scholarships", "Merit scholarships of up to 50 percent tuition waiver are offered to students who score above 95 percent in the entrance exam. Need-based aid is available on request."),
    ("Hostel facilities", "Separate hostels are provided for boys and girls with Wi-Fi, a gym, laundry service and a 24-hour security desk. Hostel allotment is first come, first served."),
    ("Placements", "The placement cell hosts over 150 companies each year. The average package last year was 7 lakh rupees per annum and the highest package was 42 lakh rupees."),
    ("Courses offered", "The university offers B.Tech in Computer Science, Electronics, Mechanical and Civil Engineering, plus M.Tech, MBA and Ph.D. programmes."),
    ("Contact", "The admissions office is open Monday to Friday, 9 AM to 5 PM. You can email admissions@university.example or call the helpline."),
]
domain_docs = [Document(page_content=f"{title}: {text}", metadata={"topic": title}) for title, text in faq_data]
domain_vs = build_vectorstore(domain_docs, "domain_faq")

DOMAIN_PROMPT = PromptTemplate.from_template(
"""You are a helpful university admissions assistant. Answer the student's question using only the information below. Be short and clear.

Information:
{context}

Student question: {question}
Answer:""")

domain_chain = DOMAIN_PROMPT | llm | StrOutputParser()
OUT_OF_SCOPE_THRESHOLD = 0.75      # cosine distance; higher = less related

def domain_chatbot(question, k=2):
    results = domain_vs.similarity_search_with_score(question, k=k)   # (doc, cosine distance)
    if not results or results[0][1] > OUT_OF_SCOPE_THRESHOLD:
        return "Sorry, I can only help with university admissions questions."
    context = format_docs([doc for doc, _ in results])
    return domain_chain.invoke({"context": context, "question": question}).strip()

for q in [
    "What marks do I need to apply for B.Tech?",
    "When is the last date to apply?",
    "How much is the hostel fee?",
    "Do you give scholarships?",
    "Who won the cricket world cup?",           # out of domain
]:
    print("You:", q)
    print("Bot:", domain_chatbot(q), "\n")

You: What marks do I need to apply for B.Tech?
Bot: You are a helpful university admissions assistant. Answer the student's question using only the information below. Be short and clear.

Information:
Admission eligibility: To apply for the B.Tech programme, a candidate must have completed 12th grade with Physics, Chemistry and Mathematics and scored at least 60 percent.

Courses offered: The university offers B.Tech in Computer Science, Electronics, Mechanical and Civil Engineering, plus M.Tech, MBA and Ph.D. programmes.

Student question: What marks do I need to apply for B.Tech?
Answer: 

You: When is the last date to apply?
Bot: You are a helpful university admissions assistant. Answer the student's question using only the information below. Be short and clear.

Information:
Application deadline: The last date to submit the online application for the 2026 academic year is 30 June 2026. Late applications are accepted until 10 July with a penalty fee.

Contact: The admissions office 

In [ ]:
# OPTIONAL: chat live. Type a question and press Enter. Type "exit" to stop.
def chat_loop():
    print("University Admissions Bot (type 'exit' to quit)")
    while True:
        q = input("You: ").strip()
        if q.lower() in {"exit", "quit", ""}:
            print("Bot: Goodbye!")
            break
        print("Bot:", domain_chatbot(q), "\n")

# chat_loop()      # <- remove the leading '#' to run the interactive chat

---
## Experiment 8: Context-aware chatbot using LangChain, retrieval and an LLM
**Idea:** keep chat history. Each follow-up question (like *"Who developed it?"*) is first rewritten into a standalone question using the history, then used for retrieval and answering.

In [ ]:
class ContextAwareChatbot:
    def __init__(self, vectorstore, k=3, max_turns=4):
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        self.history = []                 # list of (user, assistant)
        self.max_turns = max_turns

        condense_prompt = PromptTemplate.from_template(
"""Given the chat history and a follow-up question, rewrite the follow-up as a complete standalone question. Replace pronouns like it, they, this or that with the actual subject.

Chat history:
{history}

Follow-up question: {question}
Standalone question:""")
        self.condense_chain = condense_prompt | llm | StrOutputParser()
        self.answer_chain = RAG_PROMPT | llm | StrOutputParser()

    def _history_text(self):
        return "\n".join(f"User: {u}\nAssistant: {a}" for u, a in self.history[-self.max_turns:])

    def chat(self, question, verbose=True):
        if self.history:
            standalone = self.condense_chain.invoke({"history": self._history_text(), "question": question}).strip()
        else:
            standalone = question
        docs = self.retriever.invoke(standalone)
        answer = self.answer_chain.invoke({"context": format_docs(docs), "question": standalone}).strip()
        self.history.append((question, answer))
        if verbose:
            print("You:", question)
            if standalone != question:
                print(f"     (understood as: {standalone})")
            print("Bot:", answer, "\n")
        return answer

    def reset(self):
        self.history = []

kb8 = build_vectorstore(split_documents(load_folder(SAMPLE_DIR)), "exp8")
bot8 = ContextAwareChatbot(kb8)

# Multi-turn conversation: later questions depend on earlier ones
bot8.chat("What is FAISS?")
bot8.chat("Who developed it?")
bot8.chat("What is ChromaDB?")
bot8.chat("Is it open source?")

print("--- New topic, new conversation ---\n")
bot8.reset()
bot8.chat("What is the Paris Agreement?")
bot8.chat("What temperature does it aim to limit warming to?")

You: What is FAISS?
Bot: Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
FAISS stands for Facebook AI Similarity Search. It is an open-source library developed by Meta AI for fast similarity search over dense vectors, and it supports GPU acceleration. ChromaDB is an open-source embedding database that stores documents, embeddings and metadata, and it is popular for building LLM applications. Other vector databases include Pinecone, Weaviate and Milvus.

The Paris Agreement, adopted in 2015, is an international treaty on climate change. Its goal is to limit global warming to well below 2 degrees Celsius above pre-industrial levels, and to pursue efforts to limit the increase to 1.5 degrees Celsius.

In supervised learning, a model is trained on labeled examples, such as emails marked as spam or not spam. Common supervised algorithms include linear regression, decision trees and support vector machines. In unsupervised 

KeyboardInterrupt: 

---
## Experiment 9: Simple AI assistant that answers using external documents
**Idea:** point the assistant at a folder of documents (`.txt` / `.pdf`). It indexes them and answers questions with the source file shown.

To use your own files, upload them in the Colab file panel (left sidebar) into a folder and set `DOCS_FOLDER` to it.

In [ ]:
DOCS_FOLDER = SAMPLE_DIR        # change to e.g. "/content/my_docs" to use your own files

class SimpleDocAssistant:
    def __init__(self, folder, k=3):
        docs = load_folder(folder)
        if not docs:
            raise ValueError(f"No .txt or .pdf files found in {folder}")
        chunks = split_documents(docs)
        self.vs = build_vectorstore(chunks, "exp9")
        self.retriever = self.vs.as_retriever(search_kwargs={"k": k})
        self.chain = RAG_PROMPT | llm | StrOutputParser()
        print(f"Assistant ready: {len({d.metadata['doc_name'] for d in docs})} file(s), {len(chunks)} chunks indexed.")

    def ask(self, question):
        docs = self.retriever.invoke(question)
        answer = self.chain.invoke({"context": format_docs(docs), "question": question}).strip()
        sources = sorted({d.metadata["doc_name"] for d in docs})
        print("Q:", question)
        print("A:", answer)
        print("Sources:", ", ".join(sources), "\n")
        return answer, sources

assistant = SimpleDocAssistant(DOCS_FOLDER)

assistant.ask("What are the three types of machine learning?")
assistant.ask("Which energy sources do not release carbon dioxide?")
assistant.ask("What is a good chunk size for RAG?")
assistant.ask("What is the capital of Australia?")        # not in the documents

Assistant ready: 3 file(s), 13 chunks indexed.
Q: What are the three types of machine learning?
A: Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning and reinforcement learning.

In supervised learning, a model is trained on labeled examples, such as emails marked as spam or not spam. Common supervised algorithms include linear regression, decision trees and support vector machines. In unsupervised learning, the model finds structure in unlabeled data, for example by clustering customers into groups using the k-means algorithm.

Reinforcement learning trains an agent to make decisions by rewarding good actions and penalising bad ones. It is used in robotics and game playing. Deep learning is

('Answer the question using only the context below. If the answer is not in the context, say "I don\'t know".\n\nContext:\nThe effects of climate change include rising sea levels, more frequent heatwaves, stronger storms and changes in rainfall patterns. Coastal cities and small island nations are especially vulnerable to rising sea levels.\n\nThe Paris Agreement, adopted in 2015, is an international treaty on climate change. Its goal is to limit global warming to well below 2 degrees Celsius above pre-industrial levels, and to pursue efforts to limit the increase to 1.5 degrees Celsius.\n\nFAISS stands for Facebook AI Similarity Search. It is an open-source library developed by Meta AI for fast similarity search over dense vectors, and it supports GPU acceleration. ChromaDB is an open-source embedding database that stores documents, embeddings and metadata, and it is popular for building LLM applications. Other vector databases include Pinecone, Weaviate and Milvus.\n\nQuestion: What 

---
## Experiment 10: Document-based AI assistant with multiple uploaded documents
**Features:** upload several files, list them, ask across all documents or restrict a question to one document, see sources and page numbers, and remove a document.

Set `UPLOAD_FILES = True` below to open the Colab upload dialog (select one or more `.pdf` / `.txt` files).

In [ ]:
UPLOAD_FILES = False            # <- set True to upload your own PDFs / text files
UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

if UPLOAD_FILES:
    from google.colab import files
    uploaded = files.upload()                                   # opens the file picker
    for fname in uploaded:
        shutil.move(fname, os.path.join(UPLOAD_DIR, fname))
    print("Uploaded:", list(uploaded))

# Fall back to the sample documents if nothing was uploaded
paths = sorted(p for p in glob.glob(os.path.join(UPLOAD_DIR, "*")) if p.lower().endswith((".txt", ".pdf")))
if not paths:
    print("No uploads found - using the sample documents instead.")
    paths = sorted(glob.glob(os.path.join(SAMPLE_DIR, "*.txt")))
print("Files to index:", [os.path.basename(p) for p in paths])

No uploads found - using the sample documents instead.
Files to index: ['climate_change.txt', 'machine_learning.txt', 'rag_and_vector_databases.txt']


In [ ]:
class MultiDocAssistant:
    def __init__(self, k=4):
        self.vs = new_vectorstore("exp10")
        self.k = k
        self.chain = RAG_PROMPT | llm | StrOutputParser()
        self.doc_chunks = {}                                    # doc_name -> number of chunks

    def add_file(self, path):
        chunks = split_documents(load_file(path))
        self.vs.add_documents(chunks)
        name = os.path.basename(path)
        self.doc_chunks[name] = self.doc_chunks.get(name, 0) + len(chunks)
        print(f"Added {name}: {len(chunks)} chunks")

    def list_documents(self):
        print("Documents in the assistant:")
        for name, n in self.doc_chunks.items():
            print(f"  - {name} ({n} chunks)")

    def remove_document(self, name):
        self.vs._collection.delete(where={"doc_name": name})
        self.doc_chunks.pop(name, None)
        print("Removed", name)

    def ask(self, question, only_doc=None):
        search_kwargs = {"k": self.k}
        if only_doc:
            search_kwargs["filter"] = {"doc_name": only_doc}
        docs = self.vs.as_retriever(search_kwargs=search_kwargs).invoke(question)
        context = "\n\n".join(f"[{d.metadata['doc_name']}] {d.page_content}" for d in docs)
        answer = self.chain.invoke({"context": context, "question": question}).strip()
        cites = sorted({
            d.metadata["doc_name"] + (f" (page {d.metadata['page'] + 1})" if "page" in d.metadata else "")
            for d in docs
        })
        print("Q:", question + (f"   [restricted to {only_doc}]" if only_doc else ""))
        print("A:", answer)
        print("Sources:", ", ".join(cites), "\n")
        return answer, cites

multi = MultiDocAssistant()
for p in paths:
    multi.add_file(p)
print()
multi.list_documents()

Added climate_change.txt: 4 chunks
Added machine_learning.txt: 4 chunks
Added rag_and_vector_databases.txt: 5 chunks

Documents in the assistant:
  - climate_change.txt (4 chunks)
  - machine_learning.txt (4 chunks)
  - rag_and_vector_databases.txt (5 chunks)


In [ ]:
# Ask across ALL documents (the assistant finds the right file by itself)
multi.ask("What is the Paris Agreement?")
multi.ask("Who introduced RAG?")
multi.ask("What is reinforcement learning?")

# Restrict a question to ONE document using a metadata filter
first_doc = list(multi.doc_chunks)[0]
multi.ask("Summarise the main topic of this document.", only_doc=first_doc)

# Remove a document and confirm it is gone
last_doc = list(multi.doc_chunks)[-1]
multi.remove_document(last_doc)
multi.list_documents()

Q: What is the Paris Agreement?
A: Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
[climate_change.txt] The Paris Agreement, adopted in 2015, is an international treaty on climate change. Its goal is to limit global warming to well below 2 degrees Celsius above pre-industrial levels, and to pursue efforts to limit the increase to 1.5 degrees Celsius.

[rag_and_vector_databases.txt] FAISS stands for Facebook AI Similarity Search. It is an open-source library developed by Meta AI for fast similarity search over dense vectors, and it supports GPU acceleration. ChromaDB is an open-source embedding database that stores documents, embeddings and metadata, and it is popular for building LLM applications. Other vector databases include Pinecone, Weaviate and Milvus.

[climate_change.txt] The effects of climate change include rising sea levels, more frequent heatwaves, stronger storms and changes in rainfall patterns. Coastal 

In [ ]:
# OPTIONAL: interactive Q&A over your documents. Type "exit" to stop.
def multi_chat():
    print("Ask questions about your documents (type 'exit' to quit).")
    multi.list_documents()
    while True:
        q = input("\nYou: ").strip()
        if q.lower() in {"exit", "quit", ""}:
            break
        multi.ask(q)

# multi_chat()      # <- remove the leading '#' to run

---
### Done
You now have: embeddings and similarity search (1-2), vector databases (3-4), RAG QA and an end-to-end pipeline (5-6), LangChain chatbots (7-8) and document assistants (9-10).

**Tips**
- Better answers: set `LLM_NAME = "google/flan-t5-large"` in the setup cell and use a GPU runtime (*Runtime > Change runtime type > T4 GPU*).
- Want a stronger LLM? Replace `llm` / `llm_generate` with `ChatOpenAI`, `ChatGoogleGenerativeAI` or `ChatAnthropic` from LangChain; the rest of the code stays the same.